In [2]:
import os
from dotenv import load_dotenv
from typing import TypedDict, Annotated, Sequence
import operator

# Core LangGraph components
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

# Memory and LLM
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage, AIMessageChunk

# Load API keys and set up tracing
load_dotenv()
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGSMITH_PROJECT"] = "Intro to LangGraph"

# Define the State
class GraphState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

# Define the Node
def call_llm(state: GraphState):
    print("--- Calling LLM ---")
    llm = ChatOpenAI(model="gpt-4o")
    # Using .stream() on the LLM provides token-by-token output
    response_stream = llm.stream(state['messages'])

    # FIX: Iterate through the stream and yield individual chunks
    # 'add_messages' knows how to aggregate AIMessageChunk objects
    for chunk in response_stream:
        yield {"messages": [chunk]}

# Build the Graph
workflow = StateGraph(GraphState)
workflow.add_node("llm", call_llm)
workflow.add_edge(START, "llm")
workflow.add_edge("llm", END)

# Compile the graph
app = workflow.compile()

# --- Run the Graph with Streaming ---
print("--- Streaming LLM Response ---")
inputs = {"messages": [HumanMessage(content="Write a short haiku about LangGraph.")]}

# Use app.stream() with stream_mode="updates" to get intermediate chunks
for event in app.stream(inputs, stream_mode="updates"):
    # The event dictionary contains the node name and its streaming output
    for node_name, output_chunk in event.items():
        if "messages" in output_chunk:
            # Print the content of each streamed chunk as it arrives
            for message_chunk in output_chunk["messages"]:
                if hasattr(message_chunk, 'content'):
                    print(message_chunk.content, end="", flush=True)

print("\n--- Streaming Complete ---")

--- Streaming LLM Response ---
--- Calling LLM ---

--- Streaming Complete ---
